# **Generalized Linear Models**

### **Components:**

- Random Component
- Systematic Component
- Link Function

### **Common Distributions:**

- **Normal Distribution:**
	- Continuous Data
	- Accept All Possible Value in (-inf, +inf)
	- Bell Shape Curve Centered Aroung the Mean
	- Constant Variance
 
- **Binomial Distribution:**
	- Binary or Porportion Data

- **Poisson Distribution:**
	- Count Data
	- Discrete
	- Right-Skewed
	- Suitable for Counting Rare Events

- **Negative Binomial Distribution:**
	- Count Data
	- Discrete
	- Right-Skewed
	- More Flexible than Poisson
	- Suitable for Count Data with Overdispersion

- **Gamma Distribution:**
	- Continuous
	- Right-Skewed

- **Log-Transformed Normal:**
	- Continuous
	- Right-Skewed

### **Common Link Functions:**

- **Identity:**
	- for Normal Distribution
	- When the random variable can take any value in the real number line

- **Logit:**
	- for Binary Classification
	- When the random variable is a probability or proportion

- **Log:**
	- for Poisson Distribution, Gamma Distribution
	- When the random variable is positive

- **Inverse:**
	- for Gamma Distribution
	- When the random variable is positive and the effect of predictors are multiplicative

### **Approach:**

1. Plot the distribution of the target variable and compare it to distributions plots. Collect knowledge about the target variable (for example it's a counting data, binary data, ...). Select best suited distribution/distributions for the target variable. At the end you should compare the results for different random variables.

2. Select suitable link function.

### **Models:**

- Linear Regression
- Logistic Regression ✅
- Probit Regression ✅
- Ordinal Regression
- Poisson Regression
- Gamma Regression
- ...

In [1]:
# study different GLMs implemented in scikit-learn

## **Logistic Regression:**

- **Random Component:** Bernoulli or Binomial (for multinomial classification) Distribution
- **Link Function:** Logit Function
- **Systematic Component:** Linear Predictor

This king of GLM is suitable for classifying the binary target variable or we can use some techniques to do multinomial calssification using logistic regression.

**Multinomial Classification Approaches:**
- One Vs. Rest Approach
- Softmax Approach

In [58]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, validation_curve, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.multiclass import OneVsRestClassifier


In [52]:
data = load_breast_cancer()
X, y = data.data, data.target

assert X.shape == (569, 30)
assert y.shape == (569, )
print("data loaded successfully.")

data loaded successfully.


In [20]:
df = pd.concat(
    [
        pd.DataFrame(X, columns=data.feature_names),
        pd.Series(y, name="target")
	],
    axis=1
)
df

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,radius error,texture error,perimeter error,area error,smoothness error,compactness error,concavity error,concave points error,symmetry error,fractal dimension error,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,1.0950,0.9053,8.589,153.40,0.006399,0.04904,0.05373,0.01587,0.03003,0.006193,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,0.5435,0.7339,3.398,74.08,0.005225,0.01308,0.01860,0.01340,0.01389,0.003532,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,0.7456,0.7869,4.585,94.03,0.006150,0.04006,0.03832,0.02058,0.02250,0.004571,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,0.4956,1.1560,3.445,27.23,0.009110,0.07458,0.05661,0.01867,0.05963,0.009208,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,0.7572,0.7813,5.438,94.44,0.011490,0.02461,0.05688,0.01885,0.01756,0.005115,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,1.1760,1.2560,7.673,158.70,0.010300,0.02891,0.05198,0.02454,0.01114,0.004239,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115,0
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,0.7655,2.4630,5.203,99.04,0.005769,0.02423,0.03950,0.01678,0.01898,0.002498,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637,0
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,0.4564,1.0750,3.425,48.55,0.005903,0.03731,0.04730,0.01557,0.01318,0.003892,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820,0
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,0.7260,1.5950,5.772,86.22,0.006522,0.06158,0.07117,0.01664,0.02324,0.006185,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400,0


In [21]:
# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# scaling the data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [23]:
# model training 
logreg = LogisticRegression(
    C=np.inf,
    class_weight=None, # set it to 'balanced' to compare the results
    random_state=42,
    l1_ratio=0, # by default it has l2 regularization
)

logreg.fit(X_train_scaled, y_train)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",inf
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver 

In [35]:
sign_status = []
for coef in logreg.coef_.ravel():
    if coef > 0:
        sign_status.append("positive")
    elif coef == 0:
        sign_status.append("zero")
    elif coef < 0:
        sign_status.append("negative")

coefficient_df = pd.DataFrame(
    data=abs(logreg.coef_.ravel()), columns=["coef"]
)
coefficient_df['sign'] = sign_status
coefficient_df.sort_values(by="coef", ascending=False).reset_index(names="feature_ID")

,feature_ID,coef,sign
0,5,274.940098,positive
1,7,266.184736,negative
2,10,259.639462,negative
3,16,246.608093,positive
4,26,244.059711,negative
5,28,169.654855,negative
6,9,168.017946,negative
7,13,167.389401,negative
8,6,134.838158,negative
9,19,131.939234,positive


In [38]:
# model training 
logreg_l2 = LogisticRegression(
    C=1, # inverse regularization strength
    class_weight=None, # set it to 'balanced' to compare the results
    random_state=42,
    l1_ratio=0, # by default it has l2 regularization
)

logreg_l2.fit(X_train_scaled, y_train)

sign_status = []
for coef in logreg_l2.coef_.ravel():
    if coef > 0:
        sign_status.append("positive")
    elif coef == 0:
        sign_status.append("zero")
    elif coef < 0:
        sign_status.append("negative")

coefficient_df = pd.DataFrame(
    data=abs(logreg_l2.coef_.ravel()), columns=["coef"]
)
coefficient_df['sign'] = sign_status
coefficient_df.sort_values(by="coef", ascending=False).reset_index(names="feature_ID")

,feature_ID,coef,sign
0,21,1.350606,negative
1,10,1.268178,negative
2,28,1.208200,negative
3,7,1.119804,negative
4,26,0.943053,negative
5,13,0.907186,negative
6,20,0.879840,negative
7,23,0.841846,negative
8,6,0.801458,negative
9,27,0.778217,negative


In [42]:
# model training 
logreg_l1 = LogisticRegression(
    C=1, # inverse regularization strength
    class_weight=None, # set it to 'balanced' to compare the results
    random_state=42,
    l1_ratio=1, # by default it has l2 regularization
    solver="liblinear"
)

logreg_l1.fit(X_train_scaled, y_train)

sign_status = []
for coef in logreg_l1.coef_.ravel():
    if coef > 0:
        sign_status.append("positive")
    elif coef == 0:
        sign_status.append("zero")
    elif coef < 0:
        sign_status.append("negative")

coefficient_df = pd.DataFrame(
    data=abs(logreg_l1.coef_.ravel()), columns=["coef"]
)
coefficient_df['sign'] = sign_status
coefficient_df.sort_values(by="coef", ascending=False).reset_index(names="feature_ID")

,feature_ID,coef,sign
0,23,3.122343,negative
1,10,2.470136,negative
2,7,2.437017,negative
3,21,1.854182,negative
4,26,1.298469,negative
5,28,1.032401,negative
6,15,0.894132,positive
7,20,0.762839,negative
8,18,0.459953,positive
9,14,0.432847,negative


In [49]:
y_pred = logreg.predict(X_test_scaled)
y_pred_l2 = logreg_l2.predict(X_test_scaled)
y_pred_l1 = logreg_l1.predict(X_test_scaled)

score = f1_score(y_test, y_pred)
score_l2 = f1_score(y_test, y_pred_l2)
score_l1 = f1_score(y_test, y_pred_l1)

score, score_l2, score_l1

(0.948905109489051, 0.9790209790209791, 0.9787234042553191)

In [54]:
# multiclass classification using logistic regression (one vs. rest approach)
# loading data
data = load_iris()
X, y = data.data, data.target

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=42, random_state=42
)

# scaling inputs
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# model training
loreg = LogisticRegression(
    random_state=42
)
ovr_clf = OneVsRestClassifier(estimator=logreg, n_jobs=-1)

ovr_clf.fit(X_train_scaled, y_train)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...ndom_state=42)
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",-1
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
Name,Type,Value
"classes_ classes_: array, shape = [`n_classes`]Class labels.","ndarray[int64](3,)","[0,1,2]"
estimators_ estimators_: list of `n_classes` estimatorsEstimators used for predictions.,list,"[LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42), LogisticRegre...ndom_state=42)]"
label_binarizer_ label_binarizer_: LabelBinarizer objectObject used to transform multiclass labels to binary labels andvice-versa.,LabelBinarizer,LabelBinarize...e_output=True)
multilabel_ multilabel_: booleanWhether a OneVsRestClassifier is a multilabel classifier.,bool,False
n_classes_ n_classes_: intNumber of classes.,int,3
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 0.24,int,4
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",inf


In [57]:
# multiclass classification using logistic regression (softmax approach)
# loading data
data = load_iris()
X, y = data.data, data.target

# train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=42, random_state=42
)

# scaling inputs
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# model training
loreg = LogisticRegression(
    random_state=42
)
logreg.fit(X_train_scaled, y_train)

,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",inf
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",42
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver 